In [4]:
import pickle
import numpy as np
import os

import tiledbsoma
import cellxgene_census

print(f"TileDB-SOMA version: {tiledbsoma.__version__}")
print(f"Census API version: {cellxgene_census.__version__}")

# This should now work without the encoding error
census_version =  "2025-01-30"
with cellxgene_census.open_soma(census_version=census_version) as census:
    print("Success! Census opened.")


# Output paths
output_folder = "/gladstone/theodoris/home/jgortega/2026/04_Apr/Redoing_eval_cardiomyocites/h5ad_dir"
os.makedirs(output_folder, exist_ok=True)
output_file = "regular_ventricular_cardiac_myocyte_no_decades.h5ad"
output_path = output_folder + '/' + output_file

viable_cell_types = ['cardiac muscle cell', 
                     'regular ventricular cardiac myocyte']

tissue_subset = ['heart right ventricle', 
                 'apex of heart', 
                 'heart left ventricle', 
                 'interventricular septum', 
                 'apical region of left ventricle', 
                 'heart left ventricle', 
                 'cardiac ventricle', 
                 'heart right ventricle', 
                 'anterior wall of left ventricle']

cell_metadata_to_collect = ['soma_joinid', 'dataset_id', 'assay', 'cell_type', 
                            'development_stage', 'disease', 'donor_id', 
                            'self_reported_ethnicity', 'sex', 'tissue', 'tissue_general']

# Load the development stage to time mapping dictionary
with open('dev_to_time.pkl', 'rb') as f:
    dev_to_time = pickle.load(f)

# Extract viable ages from the dictionary keys
viable_ages = list(dev_to_time.keys())

# Construct the value filter query string
value_filter = (
    f"assay != 'BD Rhapsody Targeted mRNA' and "
    f"is_primary_data == True and "
    f"disease == 'normal' and "
    f"sex in ['male', 'female'] and "
    f"cell_type in {viable_cell_types} and "
    f"tissue in {tissue_subset} and "
    f"development_stage in {viable_ages}"
)

# Query the CxG Census and extract the data
with cellxgene_census.open_soma(census_version=census_version) as census:
    print("Querying and downloading data... This may take a moment.")
    
    # get_anndata directly returns a merged AnnData object based on the filter
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens", # Required argument for get_anndata
        obs_value_filter=value_filter,
        column_names={"obs": cell_metadata_to_collect}
    )

# Convert cell_type to string and rename cardiac muscle cell to regular ventricular cardiac myocyte
adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
adata.obs.loc[adata.obs['cell_type'] == 'cardiac muscle cell', 'cell_type'] = 'regular ventricular cardiac myocyte'

# Filtering inpsecific decades 
adata = adata[~adata.obs['development_stage'].str.contains('decade', case=False, na=False), :].copy()

# Re-categorize for memory efficiency
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

# Create 'time' column using the dictionary mapping and ensure it is an integer
adata.obs['time'] = adata.obs['development_stage'].map(dev_to_time).astype(int)

# Ensure only the requested metadata columns are kept (in case extra columns tagged along)
final_obs_columns = cell_metadata_to_collect + ['time']
adata.obs = adata.obs[[col for col in final_obs_columns if col in adata.obs.columns]]

# Save to a single h5ad file
print(f"Saving {adata.n_obs} cells to {output_path}...")
adata.write_h5ad(output_path)
print("Complete!")

TileDB-SOMA version: 2.3.0
Census API version: 1.17.0
Success! Census opened.
Querying and downloading data... This may take a moment.


/tmp/ipykernel_1852649/644750439.py:64: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  adata = cellxgene_census.get_anndata(
/gladstone/theodoris/home/jgortega/miniconda3/envs/cxgdownloadenv/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/gladstone/theodoris/home/jgortega/miniconda3/envs/cxgdownloadenv/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Saving 10000 cells to /gladstone/theodoris/home/jgortega/2026/04_Apr/Redoing_eval_cardiomyocites/h5ad_dir/regular_ventricular_cardiac_myocyte_no_decades.h5ad...
Complete!
